In [13]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50
import torchvision.transforms.functional as TF
from torchvision.models.segmentation import fcn_resnet50
from torch.utils.data import DataLoader
import numpy as np
from torch.utils.data import Dataset
import torchvision.transforms as T
from tqdm import tqdm
import time
import pandas as pd
import torch.nn.utils.prune as prune
import torch.optim as optim

In [ ]:
ID2TRAINID = {
    7: 0,    # road
    8: 1,    # sidewalk
    11: 2,   # building
    12: 3,   # wall
    13: 4,   # fence
    17: 5,   # pole
    19: 6,   # traffic light
    20: 7,   # traffic sign
    21: 8,   # vegetation
    22: 9,   # terrain
    23: 10,  # sky
    24: 11,  # person
    25: 12,  # rider
    26: 13,  # car
    27: 14,  # truck
    28: 15,  # bus
    31: 16,  # train
    32: 17,  # motorcycle
    33: 18   # bicycle
}


In [5]:
"""
# ===== PATH =====
GT_FINE_ROOT = "C:\\Users\\alber\\OneDrive\\Desktop\\progetto_IMCS\\gtFine"   # cambia se serve

# ===== LABEL → TRAINID =====

def convert_label_to_train(label_path, save_path):
    label = np.array(Image.open(label_path))
    train = np.full(label.shape, 255, dtype=np.uint8)

    for label_id, train_id in ID2TRAINID.items():
        train[label == label_id] = train_id

    Image.fromarray(train).save(save_path)

# ===== CONVERSIONE =====
for split in ["train","val"]:  #"train",
    split_dir = os.path.join(GT_FINE_ROOT, split)
    print(f"\nConverting split: {split}")

    for city in os.listdir(split_dir):
        city_dir = os.path.join(split_dir, city)

        for file in tqdm(os.listdir(city_dir)):
            if file.endswith("_gtFine_labelIds.png"):
                label_path = os.path.join(city_dir, file)
                train_path = label_path.replace(
                    "_gtFine_labelIds.png",
                    "_gtFine_trainIds.png"
                )

                convert_label_to_train(label_path, train_path)
"""

'\n# ===== PATH =====\nGT_FINE_ROOT = "C:\\Users\\alber\\OneDrive\\Desktop\\progetto_IMCS\\gtFine"   # cambia se serve\n\n# ===== LABEL → TRAINID =====\n\ndef convert_label_to_train(label_path, save_path):\n    label = np.array(Image.open(label_path))\n    train = np.full(label.shape, 255, dtype=np.uint8)\n\n    for label_id, train_id in ID2TRAINID.items():\n        train[label == label_id] = train_id\n\n    Image.fromarray(train).save(save_path)\n\n# ===== CONVERSIONE =====\nfor split in ["train","val"]:  #"train",\n    split_dir = os.path.join(GT_FINE_ROOT, split)\n    print(f"\nConverting split: {split}")\n\n    for city in os.listdir(split_dir):\n        city_dir = os.path.join(split_dir, city)\n\n        for file in tqdm(os.listdir(city_dir)):\n            if file.endswith("_gtFine_labelIds.png"):\n                label_path = os.path.join(city_dir, file)\n                train_path = label_path.replace(\n                    "_gtFine_labelIds.png",\n                    "_gtFine_tr

In [6]:
class CityscapesDataset(Dataset):
    def __init__(self, img_root, mask_root, size=(384, 768),use_Train=True):
        self.img_paths = []
        self.mask_paths = []
        self.size = size

        for city in os.listdir(img_root):
            img_dir = os.path.join(img_root, city)
            for file in os.listdir(img_dir):
                if file.endswith("_leftImg8bit.png"):
                    img_path = os.path.join(img_dir, file)
                    if use_Train == True:
                        mask_path = os.path.join(
                            mask_root,
                            city,
                            file.replace("_leftImg8bit.png", "_gtFine_trainIds.png")
                        )
                    else:
                        mask_path = os.path.join(
                            mask_root,
                            city,
                            file.replace("_leftImg8bit.png", "_gtFine_labelIds.png")
                            )
                    self.img_paths.append(img_path)
                    self.mask_paths.append(mask_path)

        self.img_transform = T.Compose([
            T.Resize(size, interpolation=T.InterpolationMode.BILINEAR),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load image
        img = Image.open(self.img_paths[idx]).convert("RGB")
        img = self.img_transform(img)

        # Load mask
        mask = Image.open(self.mask_paths[idx])
        mask = T.functional.resize(
            mask,
            self.size,
            interpolation=T.InterpolationMode.NEAREST
        )
        mask = torch.from_numpy(np.array(mask)).long()

        return img, mask


##Creare il dataloader

In [7]:
train_dataset = CityscapesDataset(
    img_root="C:\\Users\\alber\\OneDrive\\Desktop\\progetto_IMCS\\leftImg8bit\\train",
    mask_root="C:\\Users\\alber\\OneDrive\\Desktop\\progetto_IMCS\\gtFine\\train",
    size=(384, 768),
    use_Train=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


In [8]:
val_dataset = CityscapesDataset(
    img_root="C:\\Users\\alber\\OneDrive\\Desktop\\progetto_IMCS\\leftImg8bit\\val",
    mask_root="C:\\Users\\alber\\OneDrive\\Desktop\\progetto_IMCS\\gtFine\\val",
    size=(384, 768),
    use_Train=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


##Metriche

In [ ]:
DEVICE =  "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 19   # cambia se necessario
print(DEVICE)


In [10]:
def fast_hist(pred, label, num_classes):
    mask = (label >= 0) & (label < num_classes)
    hist = torch.bincount(
        num_classes * label[mask].long() + pred[mask].long(),
        minlength=num_classes ** 2
    ).reshape(num_classes, num_classes)
    return hist


In [11]:
def compute_metrics(confusion_matrix):
    acc = torch.diag(confusion_matrix).sum() / confusion_matrix.sum()

    iu = torch.diag(confusion_matrix) / (
        confusion_matrix.sum(1) +
        confusion_matrix.sum(0) -
        torch.diag(confusion_matrix) +
        1e-6
    )

    miou = iu.mean()
    return acc.item(), miou.item()


In [12]:
def inference_time(model, dataloader, device, warmup=10):
    model.eval()
    times = []

    with torch.no_grad():
        # warmup
        for i, (x, _) in enumerate(dataloader):
            if i == warmup:
                break
            x = x.to(device)
            _ = model(x)

        # timing
        for x, _ in dataloader:
            x = x.to(device)

            start = time.perf_counter()
            _ = model(x)
            end = time.perf_counter()

            times.append(end - start)

    return np.mean(times)


In [13]:
def evaluate_model(model, dataloader, num_classes, device):
    model.to(device)
    model.eval()

    confmat = torch.zeros((num_classes, num_classes), device=device)

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            confmat += fast_hist(preds.flatten(), y.flatten(), num_classes)

    acc, miou = compute_metrics(confmat)
    avg_time = inference_time(model, dataloader, device)

    return acc, miou, avg_time


##Let's load 

First model 

In [14]:
class ResNet50Segmentation(nn.Module):
    def __init__(self, num_classes=19, pretrained=True):
        super().__init__()

        # ===== Encoder: ResNet-50 =====
        backbone = resnet50(pretrained=pretrained)

        self.encoder = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        # ===== Segmentation head =====
        self.classifier = nn.Conv2d(
            in_channels=2048,
            out_channels=num_classes,
            kernel_size=1
        )

    def forward(self, x):
        input_size = x.shape[-2:]  # (H, W)

        features = self.encoder(x)        # (B, 2048, H/32, W/32)
        logits = self.classifier(features)  # (B, C, H/32, W/32)

        # Upsample to original resolution
        logits = F.interpolate(
            logits,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        return logits


In [15]:
modello_resnet50 = ResNet50Segmentation(num_classes=NUM_CLASSES, pretrained=False)
modello_resnet50.to(DEVICE)
modello_resnet50.load_state_dict(torch.load("modello_base.pth", map_location=DEVICE))

c:\Users\alber\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\alber\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
C:\Users\alber\AppData\Local\Temp\ipykernel_2388\1814424169.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future r

<All keys matched successfully>

##UNET model 

In [16]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


In [17]:
class UNetResNet50(nn.Module):
    def __init__(self, num_classes=19, pretrained=True):
        super().__init__()

        backbone = resnet50(pretrained=pretrained)

        # Encoder
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1  # 256
        self.layer2 = backbone.layer2  # 512
        self.layer3 = backbone.layer3  # 1024
        self.layer4 = backbone.layer4  # 2048

        # Decoder
        self.dec4 = DecoderBlock(2048, 1024, 512)
        self.dec3 = DecoderBlock(512, 512, 256)
        self.dec2 = DecoderBlock(256, 256, 128)

        self.final = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, 1)
        )

    def forward(self, x):
        input_size = x.shape[-2:]

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        e1 = self.layer1(x)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        d4 = self.dec4(e4, e3)
        d3 = self.dec3(d4, e2)
        d2 = self.dec2(d3, e1)

        out = F.interpolate(d2, size=input_size, mode="bilinear", align_corners=False)
        return self.final(out)


In [18]:
modello_UNET = UNetResNet50(num_classes=NUM_CLASSES, pretrained=False)
modello_UNET.to(DEVICE)
modello_UNET.load_state_dict(torch.load("unet_epoch_10.pth", map_location=DEVICE))

C:\Users\alber\AppData\Local\Temp\ipykernel_2388\2156138514.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modello_UNET.load_state_dict(torch.load("unet_epoch_10.pth", 

<All keys matched successfully>

##Lottery ticket 

In [19]:
import torch.nn.utils.prune as prune

In [20]:
modello_lottery_ticket = UNetResNet50(num_classes=NUM_CLASSES, pretrained=False)
modello_lottery_ticket.to(DEVICE)
parameters_to_prune = [
        (m, "weight") for m in modello_lottery_ticket.modules()
        if isinstance(m, nn.Conv2d)
    ]
modello_lottery_ticket.load_state_dict(torch.load("modello_iniziale.pth", map_location=DEVICE))

C:\Users\alber\AppData\Local\Temp\ipykernel_2388\3628138928.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modello_lottery_ticket.load_state_dict(torch.load("modello_in

<All keys matched successfully>

In [21]:
print(parameters_to_prune)

[(Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False), 'weight'), (Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False), 'weight'), (Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False), 'weight'), (Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False), 'weight'), (Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1), bias=False), 'weight'), (Conv2d(128, 128, kernel_siz

In [22]:
    for giro in range(12):
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=0.2)
        load_model_path=f"lottery_ticket_round_{giro}.pth"
        state_dict = torch.load(load_model_path, map_location=DEVICE)
        modello_lottery_ticket.load_state_dict(state_dict)
        print(f"caricato{load_model_path}")

C:\Users\alber\AppData\Local\Temp\ipykernel_2388\2126598970.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(load_model_path, map_location=DEVICE)

caricatolottery_ticket_round_0.pth
caricatolottery_ticket_round_1.pth
caricatolottery_ticket_round_2.pth
caricatolottery_ticket_round_3.pth
caricatolottery_ticket_round_4.pth
caricatolottery_ticket_round_5.pth
caricatolottery_ticket_round_6.pth
caricatolottery_ticket_round_7.pth
caricatolottery_ticket_round_8.pth
caricatolottery_ticket_round_9.pth
caricatolottery_ticket_round_10.pth
caricatolottery_ticket_round_11.pth


In [23]:
def compute_sparsity(model):
    zeros = 0
    total = 0
    for p in model.parameters():
        zeros += (p == 0).sum().item()
        total += p.numel()
    return zeros / total


In [24]:
print(f"{compute_sparsity(modello_lottery_ticket)*100}%")

93.00347223847723%


##modello structured pruning

In [25]:
model_pruned=UNetResNet50(num_classes=NUM_CLASSES, pretrained=False)
model_pruned.to(DEVICE)
model_pruned.load_state_dict(torch.load("unet_epoch_10.pth", map_location=DEVICE))

C:\Users\alber\AppData\Local\Temp\ipykernel_2388\277237733.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_pruned.load_state_dict(torch.load("unet_epoch_10.pth", m

<All keys matched successfully>

In [26]:
def structured_filter_pruning(
    model,
    amount=0.2,
    n=1,
    dim=0
):
    """
    amount: percentuale di filtri da rimuovere
    n: norma Ln (1 = L1, 2 = L2)
    dim: dimensione del pruning
         0 = output channels (filter pruning)
    """
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            prune.ln_structured(
                module,
                name="weight",
                amount=amount,
                n=n,
                dim=dim
            )
            prune.remove(module, "weight")


In [27]:
structured_filter_pruning(
    model_pruned,
    amount=0.7,
    n=1)

In [28]:
model_pruned.load_state_dict(torch.load("fine_tuned_pruned5.pth", map_location=DEVICE))

C:\Users\alber\AppData\Local\Temp\ipykernel_2388\320497657.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_pruned.load_state_dict(torch.load("fine_tuned_pruned5.pt

<All keys matched successfully>

In [29]:
(model_pruned.state_dict())

OrderedDict([('conv1.weight',
              tensor([[[[-2.2332e-02, -3.8785e-02, -7.8968e-02,  ..., -6.6352e-02,
                         -4.0441e-02, -9.9181e-04],
                        [-2.2310e-02, -2.9878e-02, -3.6808e-02,  ..., -3.6058e-02,
                         -3.8797e-02, -1.8382e-02],
                        [-2.8674e-02, -2.6093e-02, -3.2899e-02,  ..., -3.2079e-02,
                         -3.3307e-02, -4.4861e-03],
                        ...,
                        [ 3.2222e-02,  6.1363e-02,  8.0134e-02,  ...,  7.0282e-02,
                          4.6616e-02,  7.0000e-02],
                        [ 4.1167e-03,  3.9996e-02,  5.8496e-02,  ...,  3.9190e-02,
                          2.7095e-02,  4.6332e-02],
                        [-9.2733e-03,  2.7802e-02,  4.1741e-02,  ...,  2.9697e-02,
                          1.9499e-02,  3.8028e-02]],
              
                       [[-2.9013e-02, -4.2544e-02, -6.9665e-02,  ..., -7.1340e-02,
                         -4.2012

##Student model

In [30]:
class DecoderBlock2(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


In [31]:
from torchvision.models import resnet18

In [32]:
class UNetResNet18StudentSlim(nn.Module):
    def __init__(self, num_classes=19, pretrained=True):
        super().__init__()

        backbone = resnet18(pretrained=pretrained)

        # Encoder (identico)
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1  # 64
        self.layer2 = backbone.layer2  # 128
        self.layer3 = backbone.layer3  # 256
        self.layer4 = backbone.layer4  # 512

        # ✅ Decoder (MATCH checkpoint)
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)

        # ✅ Final (MATCH checkpoint)
        self.final = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

    def forward(self, x):
        input_size = x.shape[-2:]

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        e1 = self.layer1(x)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        d4 = self.dec4(e4, e3)
        d3 = self.dec3(d4, e2)
        d2 = self.dec2(d3, e1)

        out = F.interpolate(d2, size=input_size, mode="bilinear", align_corners=False)
        return self.final(out)


In [33]:
model_student = UNetResNet18StudentSlim(num_classes=19, pretrained=False).to(DEVICE)

In [34]:
model_student.load_state_dict(torch.load("student.pt", map_location=DEVICE))

C:\Users\alber\AppData\Local\Temp\ipykernel_2388\4264040903.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_student.load_state_dict(torch.load("student.pt", map_lo

<All keys matched successfully>

##Quantize model

In [35]:
import torch.quantization as tq
from torch.ao.quantization import QConfigMapping, get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

In [36]:

model_fp32 = modello_UNET
model_fp32.eval().to(DEVICE)

torch.backends.quantized.engine = "fbgemm"


In [37]:
qconfig = get_default_qconfig("fbgemm")

qconfig_mapping = (
    QConfigMapping()
    .set_module_name("conv1", qconfig)
    .set_module_name_regex(r"layer[1-4]\..*", qconfig)   # quantizza tutto dentro layer1..layer4
)

In [38]:
example_inputs = (torch.randn(1, 3, 384, 768, device=DEVICE),)

model_prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)
model_prepared.eval()


c:\Users\alber\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\ao\quantization\observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


GraphModule(
  (activation_post_process_0): HistogramObserver(min_val=inf, max_val=-inf)
  (conv1): ConvReLU2d(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): ReLU(inplace=True)
  )
  (activation_post_process_1): HistogramObserver(min_val=inf, max_val=-inf)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (activation_post_process_2): HistogramObserver(min_val=inf, max_val=-inf)
  (layer1): Module(
    (0): Module(
      (conv1): ConvReLU2d(
        (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
        (1): ReLU(inplace=True)
      )
      (conv2): ConvReLU2d(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
      )
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
      (downsample): Module(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
      )
      (relu): ReLU(inplace=True)
    )
    (1): Module(
      (

In [39]:
calibration_dataset = CityscapesDataset(
    img_root="leftImg8bit/train",
    mask_root="gtFine/train",
    size=(384, 768),
    use_Train=True
)


In [40]:
import random
NUM_CALIBRATION_SAMPLES = 100   # 50–200 OK

indices = random.sample(
    range(len(calibration_dataset)),
    NUM_CALIBRATION_SAMPLES
)

calibration_subset = torch.utils.data.Subset(calibration_dataset, indices)

In [41]:
calibration_loader = DataLoader(
    calibration_subset,
    batch_size=1,        # uguale all'inferenza reale
    shuffle=False,       # IMPORTANTISSIMO
    num_workers=2,
    pin_memory=True
)


In [ ]:
with torch.no_grad():
    for images, _ in calibration_loader:
        images = images.to(DEVICE)
        _ = model_prepared(images)


In [ ]:
model_int8 = convert_fx(model_prepared)
model_int8.eval()


In [17]:
## Pixel count per class in train folder

def count_pixels_per_class(mask_root, num_classes=19):
    """
    Count the number of pixels for each class in the training masks
    """
    pixel_counts = np.zeros(num_classes, dtype=np.int64)
    
    for city in os.listdir(mask_root):
        city_dir = os.path.join(mask_root, city)
        if not os.path.isdir(city_dir):
            continue
        
        print(f"Processing {city}...")
        files = os.listdir(city_dir)
        for i, file in enumerate(files):
            if file.endswith("_gtFine_trainIds.png"):
                mask_path = os.path.join(city_dir, file)
                mask = np.array(Image.open(mask_path))
                
                # Count pixels for each class
                for class_id in range(num_classes):
                    pixel_counts[class_id] += np.sum(mask == class_id)
            
            if (i + 1) % 50 == 0:
                print(f"  Processed {i + 1}/{len(files)} files")
    
    return pixel_counts

# Compute pixel counts for training set
print("Computing pixel counts for training set...")
train_mask_root = "C:\\Users\\alber\\OneDrive\\Desktop\\progetto_IMCS\\gtFine\\train"
pixel_counts = count_pixels_per_class(train_mask_root, num_classes=NUM_CLASSES)

# Create a dataframe for better visualization
class_names = [
    "road", "sidewalk", "building", "wall", "fence",
    "pole", "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck",
    "bus", "train", "motorcycle", "bicycle"
]

results_df = pd.DataFrame({
    "Class ID": range(NUM_CLASSES),
    "Class Name": class_names,
    "Pixel Count": pixel_counts,
    "Percentage": (pixel_counts / pixel_counts.sum() * 100).round(2)
})

print("\nPixel distribution per class:")
print(results_df.to_string(index=False))
print(f"\nTotal pixels: {pixel_counts.sum():,}")


Computing pixel counts for training set...
Processing aachen...
  Processed 50/870 files
  Processed 100/870 files
  Processed 150/870 files
  Processed 200/870 files
  Processed 250/870 files
  Processed 300/870 files
  Processed 350/870 files
  Processed 400/870 files
  Processed 450/870 files
  Processed 500/870 files
  Processed 550/870 files
  Processed 600/870 files
  Processed 650/870 files
  Processed 700/870 files
  Processed 750/870 files
  Processed 800/870 files
  Processed 850/870 files
Processing bochum...
  Processed 50/480 files
  Processed 100/480 files
  Processed 150/480 files
  Processed 200/480 files
  Processed 250/480 files
  Processed 300/480 files
  Processed 350/480 files
  Processed 400/480 files
  Processed 450/480 files
Processing bremen...
  Processed 50/1580 files
  Processed 100/1580 files
  Processed 150/1580 files
  Processed 200/1580 files
  Processed 250/1580 files
  Processed 300/1580 files
  Processed 350/1580 files
  Processed 400/1580 files
  Pro